# SUMO T자 교차로: 다중 지도 강화학습 연구

학습 5개 지도, 검증 2개 지도, 최종 시험 2개 지도를 분리합니다. Direct Rainbow-lite를 학습 시드 42~46으로 600회씩 학습하고 300회 체크포인트도 보존합니다. 동일 모델에서 300회·600회 체크포인트를 비교합니다. 센서 노이즈 학습은 포함하지 않습니다. PiCar·실제 도로 실험이 아닙니다. 중단되면 이 노트북을 다시 열고 셀을 위에서부터 실행하세요.

Drive에 체크포인트와 학습·검증 집계를 보존합니다. 원시 학습 에피소드 출력은 Colab VM에만 남고, 최종 평가의 차량·신호 기록은 Drive에도 저장합니다. 최종 시험 결과를 본 뒤 하이퍼파라미터를 바꾸면 그 시험은 새로운 독립 시험이 아닙니다.

In [ ]:
import base64, hashlib, io, zipfile, pathlib, os, sys
payload = '...'
blob = base64.b64decode(payload)
assert hashlib.sha256(blob).hexdigest() == '229a84f333ff25b09ae0923583ecce5fa9508bb098e7cc915db7a34981d858f3'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for x in z.infolist():
        assert (pathlib.Path('/content') / x.filename).resolve().is_relative_to(pathlib.Path('/content').resolve())
    z.extractall('/content')
os.chdir('/content/T_Junction_RL')
sys.path.insert(0, os.getcwd())
print('Source archive verified SHA256:', hashlib.sha256(blob).hexdigest())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q',
 'eclipse-sumo==1.28.0','sumo-data==1.28.0','traci==1.28.0','sumolib==1.28.0','libsumo==1.28.0',
 'gymnasium>=1.0,<2','PyYAML>=6,<7','scipy>=1.11,<2','pandas>=2.1,<4'],check=True)
import libsumo, torch
print('SUMO:',libsumo.getVersion(),'PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())


## Colab CPU 예비 실행. 3회 및 두 검증 지도 실행 성공을 확인합니다.

In [ ]:
from pathlib import Path
import subprocess, sys
SMOKE=Path('/content/sumo_study_smoke')
if (SMOKE/'training_status.json').exists():
    print('기존 예비 실행:',(SMOKE/'training_status.json').read_text())
else:
    subprocess.run([sys.executable,'-m','src.train','--mode','smoke','--output',str(SMOKE),
      '--training-seed','42','--algorithm','rainbow_lite','--control-mode','direct','--backend','libsumo','--device','cpu'],check=True)
import pandas as pd
v=pd.read_csv(SMOKE/'logs/validation.csv')
assert set(v['map'])=={'validation0','validation1'}
print('Smoke:',(SMOKE/'training_status.json').read_text())
display(v.groupby('map').cohort_delay_s.agg(['mean','count']))


## Drive 연결. Google 권한 요청이 나오면 직접 승인해야 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
STUDY_DRIVE=Path('/content/drive/MyDrive/SUMO_T_Junction_Study_20261104')
STUDY_DRIVE.mkdir(parents=True,exist_ok=True)
os.environ['STUDY_DRIVE']=str(STUDY_DRIVE)
print('Run directory:',STUDY_DRIVE)


## 5개 시드 학습 (각 600회). 중단 시 완료된 5회 단위 체크포인트부터 재개합니다.

In [ ]:
from src.train import train
import shutil,json
for seed in (42,43,44,45,46):
    local=Path('/content/study_runs')/f'seed{seed}'
    mirror=STUDY_DRIVE/f'seed{seed}'
    if not (local/'checkpoints/training_checkpoint.pt').exists() and (mirror/'checkpoints/training_checkpoint.pt').exists():
        shutil.copytree(mirror,local,dirs_exist_ok=True)
    checkpoint=local/'checkpoints/training_checkpoint.pt'
    if checkpoint.exists():
        train('full',local,resume=checkpoint,mirror=mirror)
    else:
        train('full',local,training_seed=seed,algorithm='rainbow_lite',control_mode='direct',
              backend='libsumo',device='cpu',mirror=mirror)
    shutil.copy2(local/'training_status.json',mirror/'training_status.json')
    print('DONE',seed,(local/'training_status.json').read_text())


## 두 미학습 지도 독립 평가. 총 1,040회. 중단되면 셀을 재실행합니다.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','scripts.evaluate_study'],check=True)
